# 🧠 Entropy-Driven CfC Context Pruning

Tek-akış pipeline — **Colab (A100) veya lokal/sunucu (H100) fark etmez, restart GEREKMEZ.**

- Çıktı kök dizini `CENG467_BASE` env'i ile belirlenir (otomatik: Colab → Drive, lokal → `./artifacts`).
- `SMOKE_TEST = True` → uçtan uca hızlı duman testi (birkaç dakika).
- `SMOKE_TEST = False` → tam koşu (gece boyu GPU).

Akış: **setup → veri → öğretmen (ΔNLL) → SBERT+Δt → proxy kalite → CfC eğitimi → değerlendirme → figürler**.

In [ ]:
# ============================================================
# Ortam kurulumu — Colab/lokal otomatik algılanır, restart yok
# ============================================================
import os, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/CENG467_Final'
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone https://github.com/Mrtuzy/CENG467_Final.git {REPO_DIR}')
    else:
        # Repo zaten varsa en güncel kodu çek — yoksa "Run all" ESKİ kodu kullanır!
        os.system(f'cd {REPO_DIR} && git pull --ff-only')
    os.environ.setdefault('CENG467_BASE', '/content/drive/MyDrive/CENG_467')
else:
    # Lokal/sunucu: notebook'u repo kökünden başlat (notebooks/ üst klasörü)
    REPO_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    os.environ.setdefault('CENG467_BASE', os.path.join(REPO_DIR, 'artifacts'))

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('IN_COLAB     =', IN_COLAB)
print('REPO_DIR     =', REPO_DIR)
print('CENG467_BASE =', os.environ['CENG467_BASE'])

In [ ]:
# Bağımlılıklar. Lokal jepa_conda ortamında zaten kuruluysa atlanabilir.
# NOT: 4-bit / bitsandbytes GEREKMEZ — öğretmen (Qwen2.5-1.5B) fp16 koşar.
if IN_COLAB:
    get_ipython().run_line_magic('pip', 'install -q -r requirements.txt')
else:
    print('Lokal ortam: requirements.txt elle kurulduysa bu hücre atlanabilir.')

In [ ]:
# ============================================================
# SMOKE TEST bayrağı — hızlı doğrulama için True, tam koşu için False
# ============================================================
SMOKE_TEST = True
SMOKE_FLAG = "--smoke_test" if SMOKE_TEST else ""

# config'ten çözülmüş çıktı yollarını al (sonuç/figür hücrelerinde kullanılır)
from config import OUTPUT_DIR, FIGURES_DIR, MODEL_DIR, TEACHER_MODEL_NAME
print("Öğretmen/üretici LLM :", TEACHER_MODEL_NAME)
print("OUTPUT_DIR           :", OUTPUT_DIR)
print("FIGURES_DIR          :", FIGURES_DIR)

## 1. Veri Hazırlığı (QReCC)

In [ ]:
get_ipython().system('python src/data_prep.py $SMOKE_FLAG')

## 2. Öğretmen Etiketleme — Cevap-koşullu Leave-One-Out ΔNLL

Her turn için: turn'ü attığımızda gold cevabın NLL'si ne kadar artıyor → o turn o kadar önemli.
Hafif Qwen2.5-1.5B-Instruct ile, generation/full-vocab KL olmadan.

In [ ]:
get_ipython().system('python src/teacher_labeling.py $SMOKE_FLAG')

import torch, gc; torch.cuda.empty_cache(); gc.collect()

## 3. SBERT Vektörizasyon + DistilGPT-2 Entropi → Δt

`surprisal_to_dt` (config.DT_MAPPING="rank") ile Δt üretilir. **Değerlendirme de aynı eşlemeyi kullanır.**

In [ ]:
get_ipython().system('python src/build_inputs.py $SMOKE_FLAG')

import torch, gc; torch.cuda.empty_cache(); gc.collect()

## 4. Proxy Kalite Kontrolü (Go / No-Go)

Δt ile öğretmen önem skorları arasındaki Spearman korelasyonu — CfC eğitmeden önce sinyal sağlamlık testi.

In [ ]:
get_ipython().system('python src/check_proxy_quality.py $SMOKE_FLAG')

## 5. CfC Ağı Eğitimi

In [ ]:
get_ipython().system('python src/train_cfc.py $SMOKE_FLAG')

import torch, gc; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# Eğitim eğrilerini göster (loss + F1)
from IPython.display import Image, display
import os
for fn in ("train_val_loss.png", "val_f1_curve.png"):
    fp = os.path.join(FIGURES_DIR, fn)
    if os.path.exists(fp):
        display(Image(filename=fp, width=600))
    else:
        print("Bulunamadı:", fp)

## 6. Değerlendirme (Full / CfC / Random / Cosine)

In [ ]:
get_ipython().system('python src/evaluate.py $SMOKE_FLAG')

import torch, gc; torch.cuda.empty_cache(); gc.collect()

## 7. Sonuçlar ve Grafikler

In [ ]:
import json, os
results_path = os.path.join(OUTPUT_DIR, 'eval_results.json')
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    all_metrics = list(next(iter(results.values())).keys())
    col_w = 12
    header = f"{'Method':<{col_w}}" + "".join(f"{m:>{col_w}}" for m in all_metrics)
    print(header); print('-' * len(header))
    for method, vals in results.items():
        row = f"{method:<{col_w}}"
        for m in all_metrics:
            v = vals.get(m, '-')
            row += f"{v:>{col_w}}" if isinstance(v, str) else f"{v:>{col_w}.4f}"
        print(row)
else:
    print('Sonuç dosyası bulunamadı:', results_path)

In [ ]:
# Tüm değerlendirme grafiklerini göster
from IPython.display import Image, display
import glob, os
figs = sorted(glob.glob(os.path.join(FIGURES_DIR, '*_comparison.png')))
if figs:
    for fp in figs:
        print('\n📊', os.path.basename(fp))
        display(Image(filename=fp, width=600))
else:
    print('Grafik bulunamadı:', FIGURES_DIR)

## 8. Ablation — TAU Eşiği Duyarlılığı (opsiyonel)

CfC pruning eşiğini sweep ederek kalite/verimlilik dengesini gösterir. Sunum için yedek figür.

In [ ]:
get_ipython().system('python src/ablation.py $SMOKE_FLAG')

import torch, gc; torch.cuda.empty_cache(); gc.collect()

In [ ]:
import json, os
from IPython.display import Image, display
abl_path = os.path.join(OUTPUT_DIR, 'ablation_results.json')
if os.path.exists(abl_path):
    with open(abl_path) as f:
        abl = json.load(f)
    print(f"{'TAU':<8} {'ROUGE-L':>10} {'Avg_Tokens':>12} {'Reduction%':>12}")
    print('-' * 46)
    for tau, vals in sorted(abl.items(), key=lambda x: float(x[0])):
        print(f"{float(tau):<8.2f} {vals['ROUGE-L']:>10.4f}"
              f" {vals['Avg_Tokens']:>12.1f} {vals['Reduction%']:>11.1f}%")
fig_path = os.path.join(FIGURES_DIR, 'ablation_tau_sweep.png')
if os.path.exists(fig_path):
    display(Image(filename=fig_path, width=650))

---
### ✅ Deney tamamlandı

Çıktılar `CENG467_BASE` altında:
- `outputs/eval_results.json` — Full / CfC / Random / Cosine karşılaştırması
- `figures/*.png` — ROUGE-L, token reduction, eğitim eğrileri (sunum figürleri)
- `models/best_cfc_model.pth` — eğitilmiş CfC

**Sunum için seç:** `rouge_l_comparison.png`, `token_reduction_comparison.png`, `train_val_loss.png`.